# Random Forest Regression

## 1. Import Libraries

In [ ]:
import pandas as pd # Load and mainpulate tabular data
import numpy as np # Numerical calculations

import matplotlib.pyplot as plt # Visualization

from sklearn.model_selection import train_test_split # Split data into train/test sets
from sklearn.compose import ColumnTransformer # Apply preprocessing by column type
from sklearn.preprocessing import OneHotEncoder # Encode categorical variables
from sklearn.pipeline import Pipeline # Combine preprocessing and model steps

from sklearn.ensemble import RandomForestRegressor # Main Day 13 Model

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score # Regression metrics




# 2. Load Data

In [ ]:
df = pd.read_csv("../data/processed/member_analysis_ready.csv") # Load feature-engineered dataset

df.head() # Preview Data

## 3. Define Target and Features

In [ ]:
target = "monthly_cost" # Regression target

drop_cols = [
    "member_id",
    "monthly_cost",
    "high_cost_member",
    "awv_completed"
]

X = df.drop(columns = drop_cols) # Feature matrix
y = df[target] # Target vector


## 4. Identify Column Types

In [ ]:
categorical_cols = X.select_dtypes(
    include = ["object", "string", "category", "bool"]
).columns.tolist() # Detect categorical columns


numeric_cols = X.select_dtypes(
    include = ["int64", "float64", "int32", "float32"]
).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

## 5. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, # Features
    y, # Target
    test_size = 0.20, # Use 20% for testing
    random_state = 42  # Reproducible split
)

## 6. Build Preprocessor

In [ ]:
preprocessor = ColumnsTransformer(
    transformers = [
        (
            "cat",
            OneHotEncoder(handle_unknown = "ignore"),
            categorical_cols
        )
    ],
    remainder = "passthrough" # Keep numeric variables unchanged
)

## 7. Build Random Forest Model

In [ ]:
rf_model = Pipeline(
    steps = [
        ("preprocessor", preprocessor), # Encode categorical variables 
        (
            "model",
            RandomForestRegressor(
                n_estimators = 300, 
                max_depth = None,
                min_samples_leaf = 5,
                random_state = 42,
                n_jobs = -1
            )
        )
    ]
)

## 8. Fit Model

In [ ]:
rf_model.fit(X_train, y_train) # Train Random Forest model

## 9. Evaluate Train and Test Performance

In [ ]:
y_train_pred = rf_model.predict(X_train) # Predict training target
y_test_pred = rf_model.predict(X_test) # Predict test target

train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

rf_results = pd.DataFrame({
    "Dataset": ["Train", "Test"],
    "MAE": [train_mae, test_mae],
    "RMSE": [train_rmse, test_rmse],
    "R2": [train_r2, test_r2]
})

rf_results

## 10. Actual vs Predicted Plot

In [ ]:
plt.figure(figsize = (8,6))

plt.scatter(y_test, y_test_pred, alpha = 0.5)

plt.xlabel("Actual Monthly Cost")
plt.ylabel("Predicted Monthly Cost")
plt.title("Random Forest: Actual vs Predicted Monthly Cost")

plt.show()

## 11. Residual Plot

In [ ]:
residuals = y_test - y_test_pred # Actual minus predicted

plt.figure(figsize=(8, 6))

plt.scatter(y_test_pred, residuals, alpha = 0.5)
plt.axhline(0, linestyle = "-")

plt.xlabel("Predicted Monthly Cost")
plt.ylabel("Residual")
plt.title("Random Forest: Residuals vs Predictede Monthly Cost")

plt.show() 



## 12. Feature Importance

In [ ]:
trained_preprocessor = rf_model.named_steps["preprocessor"] # Access fitted preprocessor
trained_rf = rf_model.named_steps["model"] # Access fitted Random Forest

feature_names = trained_preprocessor.get_feature_names_out() # Get transformed feature names 

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": trained_rf.feature_importances_
}).sort_values(by = "importance", ascending = False)

importance_df.head(20)



## 13. Plot Top Feature Importances

In [ ]:
top_features = importance_df.head(15).sort_values(by = "importance")

plt.figure(figsize = (8, 6))

plt.barh(top_features["feature"], top_features["importance"])

plt.xlabel("Feature Importance")
plt.title("Top 15 Random Forest Feature Importances")

plt.show()

## 14. Compare With Day 12 Decision Tree

In [ ]:
model_comparison = pd.DataFrame({
    "Model": [
        "Controlled Decision Tree",
        "Random Forest"
    ],
    "Test MAE": [
        463.087564,
        test_mae
    ],
    "Test RMSE": [
        732.191242,
        test_rmse
    ],
    "Test R2": [
    	0.704253,
        test_r2    
    ]
})

## 15. Findings

1. The Random Forest Regressor was used as an ensemble tree-based model for monthly cost prediction.

2. Compared with a single Decision Tree, the Random Forest averages predictions across many trees, which usually reduces overfitting.

3. The model was evaluated using MAE, RMSE, and R² on both training and test data.

4. If the Random Forest has better test RMSE or test R² than the controlled Decision Tree, that suggests the ensemble model generalized better on this synthetic dataset.

5. Feature importance values show which variables were useful for prediction in the fitted forest, but they do not prove causal effects.

6. Because the dataset is synthetic, the results reflect the assumptions built into the data-generation process rather than real-world healthcare evidence.
